In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/01_cleaned_with_risk.csv')

df.columns = df.columns.str.strip()

# ── CREATE days_to_expiry  ──
df['Expiration Date'] = pd.to_datetime(df['Expiration Date'], dayfirst=True)
today = pd.Timestamp.today()
df['days_to_expiry'] = (df['Expiration Date'] - today).dt.days

# ── TARGET VARIABLE ──
# 1 = at-risk SKU (Critical or High), 0 = safe
df['is_at_risk'] = df['Risk_Level'].isin(['Critical', 'High']).astype(int)
print(f"At-risk SKUs: {df['is_at_risk'].sum():,} ({df['is_at_risk'].mean():.1%})")


# ── FEATURE 1: Stock pressure score ──
df['stock_pressure'] = df['Stock Quantity'] / 100


# ── FEATURE 2: Expiry urgency score ──
# IMPORTANT: your data is ALL expired → adjust logic
df['expiry_urgency'] = 1 - (df['days_to_expiry'].clip(-731, 731) / 731)

# normalize so negatives don’t break scale
df['expiry_urgency'] = df['expiry_urgency'].clip(0, 1)


# ── FEATURE 3: Combined risk score ──
df['risk_score'] = df['stock_pressure'] * df['expiry_urgency']


# ── FEATURE 4: Is expired ──
df['is_expired'] = (df['days_to_expiry'] <= 0).astype(int)


# ── FEATURE 5: High stock flag ──
df['high_stock_flag'] = (df['Stock Quantity'] > 75).astype(int)


# ── FEATURE 6: Near expiry flag ──
# In your case: everything expired → this becomes always 1
df['near_expiry_flag'] = (df['days_to_expiry'] < 180).astype(int)


# ── FEATURE 7: Price tier ──
df['price_tier'] = pd.qcut(df['Price'], q=3, labels=['low','mid','high'])
df['price_tier_enc'] = df['price_tier'].map({'low':0,'mid':1,'high':2})


# ── FEATURE 8: Encode categoricals ──
df['category_enc'] = df['Product Category'].astype('category').cat.codes
df['warranty_enc'] = df['Warranty Period'].astype('category').cat.codes


# ── FEATURE 9: Financial exposure ──
df['financial_exposure'] = df['Stock Quantity'] * df['Price']


# ── FEATURE 10: Overhang ratio ──
df['overhang_ratio'] = (
    (df['Stock Quantity'] - 50).clip(0) / df['Stock Quantity'].clip(1)
)


# ── SUMMARY ──
print("\nNew features added:")
new_feats = [
    'stock_pressure','expiry_urgency','risk_score','is_expired',
    'high_stock_flag','near_expiry_flag','price_tier_enc',
    'category_enc','warranty_enc','financial_exposure','overhang_ratio'
]

print(df[new_feats].describe().round(3))


# ── SAVE ──
df.to_csv('../data/processed/02_features.csv', index=False)
print("\nSaved to data/processed/02_features.csv")

At-risk SKUs: 2,530 (25.3%)

New features added:
       stock_pressure  expiry_urgency  risk_score  is_expired  \
count       10000.000         10000.0   10000.000     10000.0   
mean            0.506             1.0       0.506         1.0   
std             0.289             0.0       0.289         0.0   
min             0.010             1.0       0.010         1.0   
25%             0.250             1.0       0.250         1.0   
50%             0.510             1.0       0.510         1.0   
75%             0.760             1.0       0.760         1.0   
max             1.000             1.0       1.000         1.0   

       high_stock_flag  near_expiry_flag  category_enc  warranty_enc  \
count        10000.000           10000.0     10000.000     10000.000   
mean             0.253               1.0         0.996         1.014   
std              0.435               0.0         0.815         0.818   
min              0.000               1.0         0.000         0.000   
25%  